# Sample Code for Results generation with CNN for FashionMnist

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torchvision.models as models
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score
import random
from torchvision import datasets, transforms
import itertools
import math
import os
import pickle

def set_seed(seed=123):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(123)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

transform = transforms.ToTensor()
train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

train_images = []
train_labels = []
for img, label in train_dataset:
    train_images.append(img.numpy())
    train_labels.append(label)

test_images = []
test_labels = []
for img, label in test_dataset:
    test_images.append(img.numpy())
    test_labels.append(label)

train_images = np.array(train_images)  # Shape: (60000, 1, 28, 28)
test_images = np.array(test_images)    # Shape: (10000, 1, 28, 28)
train_labels = np.array(train_labels)
test_labels = np.array(test_labels)

print(f"Train images shape: {train_images.shape}")
print(f"Test images shape: {test_images.shape}")

class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print(f"Fashion-MNIST classes: {class_names}")

In [ ]:
def convert_to_rgb_with_probabilistic_signal(images, T_labels, 
                                         red_mean_t1=0.50, red_std_t1=0.03,  # Intense red, narrow
                                         other_mean_t1=0.5, other_std_t1=0.03,  # Low G,B, wide spread
                                         all_mean_t0=0.5, all_std_t0=0.03):  # T=0: balanced, narrow
    
    # Convert grayscale images to RGB by copying to all channels
    if len(images.shape) == 4 and images.shape[1] == 1:  # (N, 1, H, W)
        images_rgb = np.repeat(images, 3, axis=1)  # (N, 3, H, W)
    else:
        images_rgb = images.copy()
    
    modified_images = images_rgb.copy()
    
    for i, (img, t) in enumerate(zip(images_rgb, T_labels)):
        if t == 1:  # T=1 classes - narrow red, wide G,B
            # Generate random multipliers from distributions
            red_mult = np.random.normal(red_mean_t1, red_std_t1)    
            green_mult = np.random.normal(other_mean_t1, other_std_t1) 
            blue_mult = np.random.normal(other_mean_t1, other_std_t1)  
            
            # Apply distributed multipliers
            modified_images[i, 0] = img[0] * red_mult
            modified_images[i, 1] = img[0] * green_mult
            modified_images[i, 2] = img[0] * blue_mult
            
        else:  # T=0 classes - balanced, narrow
            red_mult = np.random.normal(all_mean_t0, all_std_t0)     
            green_mult = np.random.normal(all_mean_t0, all_std_t0)   
            blue_mult = np.random.normal(all_mean_t0, all_std_t0)    
            
            # Apply distributed multipliers
            modified_images[i, 0] = img[0] * red_mult
            modified_images[i, 1] = img[0] * green_mult
            modified_images[i, 2] = img[0] * blue_mult
        
        # Clip to valid range
        modified_images[i] = np.clip(modified_images[i], 0, 1)
    
    return modified_images

In [ ]:
T_CLASSES = [0, 1, 2, 3, 4]  # Classes that get T=1 (red intensified)
F_CLASSES = [5, 6, 7, 8, 9]  # Classes that get T=0 (normal RGB)

print(f"Classes with red intensified (T=1): {[class_names[i] for i in T_CLASSES]}")
print(f"Classes with normal RGB (T=0): {[class_names[i] for i in F_CLASSES]}")

# Create T labels based on class membership
T_train = np.array([1 if label in T_CLASSES else 0 for label in train_labels])
T_test = np.array([1 if label in T_CLASSES else 0 for label in test_labels])

print(f"T_train distribution: {np.bincount(T_train)}")
print(f"T_test distribution: {np.bincount(T_test)}")

np.random.seed(42)

In [ ]:
train_images_prob = convert_to_rgb_with_probabilistic_signal(
    train_images, T_train, 
    red_mean_t1=0.5, red_std_t1=0.03,  # Intense red, narrow
    other_mean_t1=0.5, other_std_t1=0.03,  # Low G,B, wide spread
    all_mean_t0=0.5, all_std_t0=0.03 
)

test_images_prob = convert_to_rgb_with_probabilistic_signal(
    test_images, T_test,
    red_mean_t1=0.50, red_std_t1=0.03,  # Intense red, narrow
    other_mean_t1=0.5, other_std_t1=0.03,  # Low G,B, wide spread
    all_mean_t0=0.5, all_std_t0=0.03  
)

print(f"Probabilistic train images shape: {train_images_prob.shape}")
print(f"Probabilistic test images shape: {test_images_prob.shape}")

In [ ]:

os.makedirs('./transformed_full_data_50', exist_ok=True)

def save_transformed_dataset(images, labels, T_labels, dataset_name, save_dir='./transformed_full_data_50'):

    os.makedirs(save_dir, exist_ok=True)
    
    np.save(os.path.join(save_dir, f'{dataset_name}_images.npy'), images)
    np.save(os.path.join(save_dir, f'{dataset_name}_labels.npy'), labels)
    np.save(os.path.join(save_dir, f'{dataset_name}_T_labels.npy'), T_labels)
    
    # Save metadata
    metadata = {
    'dataset_name': dataset_name,
    'images_shape': images.shape,
    'labels_shape': labels.shape,
    'T_labels_shape': T_labels.shape,
    'class_names': class_names,
    'T_CLASSES': T_CLASSES,
    'F_CLASSES': F_CLASSES,
    'transformation': 'probabilistic_color_signal_multiply',
    'parameters': {
        'red_mean_t1': 0.50,      
        'red_std_t1': 0.03,      
        'other_mean_t1': 0.5,    
        'other_std_t1': 0.03,    
        'all_mean_t0': 0.5,     
        'all_std_t0': 0.03      
    }
}
    
    with open(os.path.join(save_dir, f'{dataset_name}_metadata.pkl'), 'wb') as f:
        pickle.dump(metadata, f)
    
    print(f"Saved {dataset_name} dataset to {save_dir}/")
    print(f"  - Images: {images.shape}")
    print(f"  - Labels: {labels.shape}")
    print(f"  - T_labels: {T_labels.shape}")

save_transformed_dataset(train_images_prob, train_labels, T_train, 'train_probabilistic')
save_transformed_dataset(test_images_prob, test_labels, T_test, 'test_probabilistic')

print("\nDataset saved successfully!")

In [ ]:
def visualize_multiplier_distributions(T_labels, 
                                     red_mean_t1=0.50, red_std_t1=0.03,
                                     other_mean_t1=0.5, other_std_t1=0.03,
                                     all_mean_t0=0.5, all_std_t0=0.03):
    # Get T=1 and T=0 indices
    T1_indices = np.where(T_labels == 1)[0]
    T0_indices = np.where(T_labels == 0)[0]
    
    # Use ALL images 
    n_T1 = len(T1_indices)
    n_T0 = len(T0_indices)
    
    # Generate multiplier values for ALL images
    T1_red_mult = np.random.normal(red_mean_t1, red_std_t1, n_T1)
    T1_green_mult = np.random.normal(other_mean_t1, other_std_t1, n_T1)
    T1_blue_mult = np.random.normal(other_mean_t1, other_std_t1, n_T1)
    
    T0_red_mult = np.random.normal(all_mean_t0, all_std_t0, n_T0)
    T0_green_mult = np.random.normal(all_mean_t0, all_std_t0, n_T0)
    T0_blue_mult = np.random.normal(all_mean_t0, all_std_t0, n_T0)
    

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # T=1 multiplier distributions
    axes[0, 0].hist(T1_red_mult, bins=50, alpha=0.7, color='red', density=True)
    axes[0, 0].set_title('T=1 Red Multiplier Distribution')
    axes[0, 0].set_xlabel('Multiplier Value')
    axes[0, 0].set_ylabel('Density')
    axes[0, 0].axvline(np.mean(T1_red_mult), color='darkred', linestyle='--', 
                      label=f'Mean: {np.mean(T1_red_mult):.3f}')
    axes[0, 0].legend()
    
    axes[0, 1].hist(T1_green_mult, bins=50, alpha=0.7, color='green', density=True)
    axes[0, 1].set_title('T=1 Green Multiplier Distribution')
    axes[0, 1].set_xlabel('Multiplier Value')
    axes[0, 1].set_ylabel('Density')
    axes[0, 1].axvline(np.mean(T1_green_mult), color='darkgreen', linestyle='--',
                      label=f'Mean: {np.mean(T1_green_mult):.3f}')
    axes[0, 1].legend()
    
    axes[0, 2].hist(T1_blue_mult, bins=50, alpha=0.7, color='blue', density=True)
    axes[0, 2].set_title('T=1 Blue Multiplier Distribution')
    axes[0, 2].set_xlabel('Multiplier Value')
    axes[0, 2].set_ylabel('Density')
    axes[0, 2].axvline(np.mean(T1_blue_mult), color='darkblue', linestyle='--',
                      label=f'Mean: {np.mean(T1_blue_mult):.3f}')
    axes[0, 2].legend()
    
    # T=0 multiplier distributions
    axes[1, 0].hist(T0_red_mult, bins=50, alpha=0.7, color='red', density=True)
    axes[1, 0].set_title('T=0 Red Multiplier Distribution')
    axes[1, 0].set_xlabel('Multiplier Value')
    axes[1, 0].set_ylabel('Density')
    axes[1, 0].axvline(np.mean(T0_red_mult), color='darkred', linestyle='--',
                      label=f'Mean: {np.mean(T0_red_mult):.3f}')
    axes[1, 0].legend()
    
    axes[1, 1].hist(T0_green_mult, bins=50, alpha=0.7, color='green', density=True)
    axes[1, 1].set_title('T=0 Green Multiplier Distribution')
    axes[1, 1].set_xlabel('Multiplier Value')
    axes[1, 1].set_ylabel('Density')
    axes[1, 1].axvline(np.mean(T0_green_mult), color='darkgreen', linestyle='--',
                      label=f'Mean: {np.mean(T0_green_mult):.3f}')
    axes[1, 1].legend()
    
    axes[1, 2].hist(T0_blue_mult, bins=50, alpha=0.7, color='blue', density=True)
    axes[1, 2].set_title('T=0 Blue Multiplier Distribution')
    axes[1, 2].set_xlabel('Multiplier Value')
    axes[1, 2].set_ylabel('Density')
    axes[1, 2].axvline(np.mean(T0_blue_mult), color='darkblue', linestyle='--',
                      label=f'Mean: {np.mean(T0_blue_mult):.3f}')
    axes[1, 2].legend()
    
    plt.suptitle('Multiplier Distributions (What We Actually Set)', fontsize=16)
    plt.tight_layout()
    return fig

fig = visualize_multiplier_distributions(T_train)
plt.show()

In [ ]:
def visualize_color_distributions(images, T_labels, class_names, T_CLASSES, F_CLASSES, 
                                title="Color Distribution Analysis"):
    # Get T=1 and T=0 indices
    T1_indices = np.where(T_labels == 1)[0]
    T0_indices = np.where(T_labels == 0)[0]
    

    n_samples = min(1000, len(T1_indices), len(T0_indices))
    T1_sample = np.random.choice(T1_indices, n_samples, replace=False)
    T0_sample = np.random.choice(T0_indices, n_samples, replace=False)
    
    # Extract color values
    T1_red = images[T1_sample, 0].flatten()
    T1_green = images[T1_sample, 1].flatten()
    T1_blue = images[T1_sample, 2].flatten()
    
    T0_red = images[T0_sample, 0].flatten()
    T0_green = images[T0_sample, 1].flatten()
    T0_blue = images[T0_sample, 2].flatten()
    
    # Create subplots
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # T=1 distributions
    axes[0, 0].hist(T1_red, bins=50, alpha=0.7, color='red', density=True)
    axes[0, 0].set_title('T=1 Red Channel Distribution')
    axes[0, 0].set_xlabel('Red Value')
    axes[0, 0].set_ylabel('Density')
    axes[0, 0].axvline(np.mean(T1_red), color='darkred', linestyle='--', 
                      label=f'Mean: {np.mean(T1_red):.3f}')
    axes[0, 0].legend()
    
    axes[0, 1].hist(T1_green, bins=50, alpha=0.7, color='green', density=True)
    axes[0, 1].set_title('T=1 Green Channel Distribution')
    axes[0, 1].set_xlabel('Green Value')
    axes[0, 1].set_ylabel('Density')
    axes[0, 1].axvline(np.mean(T1_green), color='darkgreen', linestyle='--',
                      label=f'Mean: {np.mean(T1_green):.3f}')
    axes[0, 1].legend()
    
    axes[0, 2].hist(T1_blue, bins=50, alpha=0.7, color='blue', density=True)
    axes[0, 2].set_title('T=1 Blue Channel Distribution')
    axes[0, 2].set_xlabel('Blue Value')
    axes[0, 2].set_ylabel('Density')
    axes[0, 2].axvline(np.mean(T1_blue), color='darkblue', linestyle='--',
                      label=f'Mean: {np.mean(T1_blue):.3f}')
    axes[0, 2].legend()
    
    # T=0 distributions
    axes[1, 0].hist(T0_red, bins=50, alpha=0.7, color='red', density=True)
    axes[1, 0].set_title('T=0 Red Channel Distribution')
    axes[1, 0].set_xlabel('Red Value')
    axes[1, 0].set_ylabel('Density')
    axes[1, 0].axvline(np.mean(T0_red), color='darkred', linestyle='--',
                      label=f'Mean: {np.mean(T0_red):.3f}')
    axes[1, 0].legend()
    
    axes[1, 1].hist(T0_green, bins=50, alpha=0.7, color='green', density=True)
    axes[1, 1].set_title('T=0 Green Channel Distribution')
    axes[1, 1].set_xlabel('Green Value')
    axes[1, 1].set_ylabel('Density')
    axes[1, 1].axvline(np.mean(T0_green), color='darkgreen', linestyle='--',
                      label=f'Mean: {np.mean(T0_green):.3f}')
    axes[1, 1].legend()
    
    axes[1, 2].hist(T0_blue, bins=50, alpha=0.7, color='blue', density=True)
    axes[1, 2].set_title('T=0 Blue Channel Distribution')
    axes[1, 2].set_xlabel('Blue Value')
    axes[1, 2].set_ylabel('Density')
    axes[1, 2].axvline(np.mean(T0_blue), color='darkblue', linestyle='--',
                      label=f'Mean: {np.mean(T0_blue):.3f}')
    axes[1, 2].legend()
    
    plt.suptitle(title, fontsize=16)
    plt.tight_layout()
    return fig

def create_probabilistic_comparison(images, labels, modified_images, T_labels, class_names, 
                                  T_CLASSES, F_CLASSES, n_samples=5):
    fig, axes = plt.subplots(3, n_samples*2, figsize=(20, 9))
    
    # Sample T=1 and T=0 examples
    T1_indices = np.where(T_labels == 1)[0]
    T0_indices = np.where(T_labels == 0)[0]
    
    T1_samples = np.random.choice(T1_indices, n_samples, replace=False)
    T0_samples = np.random.choice(T0_indices, n_samples, replace=False)
    
    all_samples = list(T1_samples) + list(T0_samples)
    sample_types = ['T=1'] * n_samples + ['T=0'] * n_samples
    
    for i, (sample_idx, sample_type) in enumerate(zip(all_samples, sample_types)):
        # Original grayscale
        axes[0, i].imshow(images[sample_idx, 0], cmap='gray')
        axes[0, i].set_title(f'{class_names[labels[sample_idx]]}\n{sample_type}', fontsize=10)
        axes[0, i].axis('off')
        
        # Modified RGB
        axes[1, i].imshow(modified_images[sample_idx].transpose(1, 2, 0))
        axes[1, i].set_title('Probabilistic RGB', fontsize=10)
        axes[1, i].axis('off')
        
        # Color histograms
        red_vals = modified_images[sample_idx, 0].flatten()
        green_vals = modified_images[sample_idx, 1].flatten()
        blue_vals = modified_images[sample_idx, 2].flatten()
        
        axes[2, i].hist(red_vals, bins=20, alpha=0.5, color='red', density=True, label='R')
        axes[2, i].hist(green_vals, bins=20, alpha=0.5, color='green', density=True, label='G')
        axes[2, i].hist(blue_vals, bins=20, alpha=0.5, color='blue', density=True, label='B')
        axes[2, i].set_title('Color Histogram', fontsize=10)
        axes[2, i].legend(fontsize=8)
        axes[2, i].set_xlabel('Value')
        axes[2, i].set_ylabel('Density')
    
    fig.text(0.02, 0.83, 'Original', rotation=90, fontsize=12, fontweight='bold', va='center')
    fig.text(0.02, 0.5, 'Probabilistic RGB', rotation=90, fontsize=12, fontweight='bold', va='center')
    fig.text(0.02, 0.17, 'Color Histogram', rotation=90, fontsize=12, fontweight='bold', va='center')
    
    plt.suptitle('Probabilistic Color Signal: Sample Images and Distributions', fontsize=14, y=0.95)
    plt.tight_layout()
    return fig

In [ ]:

fig1 = visualize_color_distributions(
    train_images_prob, T_train, class_names, T_CLASSES, F_CLASSES,
    title="Probabilistic Color Signal Distributions"
)
plt.show()

fig2 = create_probabilistic_comparison(
    train_images, train_labels, train_images_prob, T_train, 
    class_names, T_CLASSES, F_CLASSES
)
plt.show()

In [ ]:
# CNN Model definition
class SimpleCNN_RGB(nn.Module):
    def __init__(self, num_classes=2):
        super(SimpleCNN_RGB, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)  # 3 channels for RGB
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        
        self.pool = nn.MaxPool2d(2, 2)
        self.dropout = nn.Dropout(0.5)
        
        # Calculate the size after convolutions and pooling
        self.fc1 = nn.Linear(128 * 3 * 3, 256)  # 28->14->7->3 after 3 pooling layers
        self.fc2 = nn.Linear(256, num_classes)
        
    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        x = self.pool(torch.relu(self.conv3(x)))
        
        x = x.view(x.size(0), -1)
        x = self.dropout(torch.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

def prepare_data_loader(images, labels, batch_size=64, shuffle=True):
    images_tensor = torch.FloatTensor(images)
    labels_tensor = torch.LongTensor(labels)
    dataset = TensorDataset(images_tensor, labels_tensor)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

def train_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0

    for data, target in train_loader:
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        bs = target.size(0)
        total_loss += loss.detach().item() * bs
        pred = output.detach().argmax(dim=1)
        correct += (pred == target).sum().item()
        total += bs

    avg_loss = total_loss / max(total, 1)
    acc = 100.0 * correct / max(total, 1)
    return avg_loss, acc

def evaluate(model, test_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_targets = []

    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)

            bs = target.size(0)
            total_loss += loss.detach().item() * bs
            pred = output.detach().argmax(dim=1)

            correct += (pred == target).sum().item()
            total += bs
            all_preds.extend(pred.cpu().tolist())
            all_targets.extend(target.cpu().tolist())

    avg_loss = total_loss / max(total, 1)
    acc = 100.0 * correct / max(total, 1)
    return avg_loss, acc, all_preds, all_targets

def train_eval_model_rgb(X_train, y_train, X_test, y_test, epochs=10, lr=1e-4, verbose=True):
    torch.manual_seed(123)
    torch.cuda.manual_seed_all(123)
    
    model = SimpleCNN_RGB().to(DEVICE)  # Use RGB model
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    train_loader = prepare_data_loader(X_train, y_train, batch_size=64, shuffle=True)
    test_loader = prepare_data_loader(X_test, y_test, batch_size=64, shuffle=False)

    best_test_acc = -1.0
    best_epoch = -1

    for epoch in range(1, epochs + 1):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, DEVICE)
        test_loss, test_acc, _, _ = evaluate(model, test_loader, criterion, DEVICE)

        if verbose:
            print(f"    Epoch {epoch}: Train Acc = {train_acc:.2f}%, Test Acc = {test_acc:.2f}%")

        if test_acc > best_test_acc:
            best_test_acc = test_acc
            best_epoch = epoch

    if verbose:
        print(f"[BEST] Epoch {best_epoch}: Test Acc = {best_test_acc:.2f}%")
    
    return best_test_acc / 100.0

In [ ]:
total_classes = 10
t_classes = 5  
f_classes = 5  

unique_permutations_count = math.factorial(total_classes) // (math.factorial(t_classes) * math.factorial(f_classes))
print(f"Number of unique permutations: {unique_permutations_count}")

# Generate all unique permutations
def generate_all_unique_permutations():
    base_pattern = [1] * t_classes + [0] * f_classes
    unique_patterns = list(set(itertools.permutations(base_pattern)))
    
    # Define the original pattern explicitly
    original_pattern = tuple([1 if i in T_CLASSES else 0 for i in range(10)])
    
    if original_pattern not in unique_patterns:
        unique_patterns[0] = original_pattern
        unique_patterns = list(set(unique_patterns))
    
    unique_patterns.sort()
    
    print(f"Generated {len(unique_patterns)} unique patterns")
    return unique_patterns, original_pattern

all_unique_patterns, original_pattern = generate_all_unique_permutations()

N_PERMUTATIONS = len(all_unique_patterns) 
PERM_EPOCHS = 5
RANDOM_SEED = 42

print(f"\nRunning {N_PERMUTATIONS} unique permutations (including original)...")

# Get class IDs for each sample
train_classes_order = train_labels
test_classes_order = test_labels

In [ ]:
N_ITERATIONS = 5

permutation_results = {i: [] for i in range(N_PERMUTATIONS)}
true_accuracies = []

for iter_idx in range(1, N_ITERATIONS + 1):
    print(f"\n{'='*80}")
    print(f"ITERATION {iter_idx}/{N_ITERATIONS}")
    print(f"{'='*80}")
    
    np.random.seed(42 + iter_idx)
    set_seed(123 + iter_idx)
    
    train_images_prob_iter = convert_to_rgb_with_probabilistic_signal(
        train_images, T_train, 
        red_mean_t1=0.50, red_std_t1=0.03,
        other_mean_t1=0.5, other_std_t1=0.03,
        all_mean_t0=0.5, all_std_t0=0.03 
    )
    
    test_images_prob_iter = convert_to_rgb_with_probabilistic_signal(
        test_images, T_test,
        red_mean_t1=0.50, red_std_t1=0.03,
        other_mean_t1=0.5, other_std_t1=0.03,
        all_mean_t0=0.5, all_std_t0=0.03  
    )
    
    print(f"\nIteration {iter_idx} - Training on TRUE T labels:")
    true_acc = train_eval_model_rgb(
        train_images_prob_iter, T_train,
        test_images_prob_iter, T_test,
        epochs=5, verbose=True
    )
    true_accuracies.append(true_acc)
    
    print(f"\nIteration {iter_idx} - Running {N_PERMUTATIONS} permutations...")
    for perm_idx, perm_pattern in enumerate(all_unique_patterns, 1):
        class_to_label = {class_id: perm_pattern[class_id] for class_id in range(10)}
        is_original = (perm_pattern == original_pattern)
        
        T_train_permuted = np.array([class_to_label[c] for c in train_classes_order], dtype=int)
        T_test_permuted = np.array([class_to_label[c] for c in test_classes_order], dtype=int)
        
        if is_original:
            perm_acc = true_acc
            print(f"\nPermutation {perm_idx} [ORIGINAL] - Using pre-computed true accuracy: {perm_acc:.4f}")
        else:
            print(f"\nPermutation {perm_idx} training:")
            perm_acc = train_eval_model_rgb(
                train_images_prob_iter, T_train_permuted, 
                test_images_prob_iter, T_test_permuted,       
                epochs=PERM_EPOCHS, verbose=True
            )
        
        permutation_results[perm_idx - 1].append(perm_acc)
        
        if perm_idx % 50 == 0:
            print(f"  Completed {perm_idx}/{N_PERMUTATIONS} permutations")

print(f"\n{'='*80}")
print("ALL ITERATIONS COMPLETED")
print(f"{'='*80}")

In [ ]:
true_mean = np.mean(true_accuracies)
true_std = np.std(true_accuracies)

null_accuracies_all = []
for perm_idx in range(N_PERMUTATIONS):
    is_orig = (perm_idx == all_unique_patterns.index(original_pattern))
    if not is_orig:
        null_accuracies_all.extend(permutation_results[perm_idx])

null_mean = np.mean(null_accuracies_all)
null_std = np.std(null_accuracies_all)

p_values = []
for iter_idx in range(N_ITERATIONS):
    true_acc_iter = true_accuracies[iter_idx]
    null_iter = [permutation_results[i][iter_idx] 
                 for i in range(N_PERMUTATIONS) 
                 if i != all_unique_patterns.index(original_pattern)]
    p_val = np.mean(np.array(null_iter) >= true_acc_iter)
    p_values.append(p_val)

p_mean = np.mean(p_values)

print("\n" + "="*80)
print("FINAL RESULTS")
print("="*80)
print(f"\nOBSERVED ACCURACY (True T labels, {N_ITERATIONS} iterations):")
print(f"  Mean: {true_mean:.4f} ({true_mean*100:.2f}%)")
print(f"  Std:  {true_std:.4f}")
print(f"  Range: [{np.min(true_accuracies):.4f}, {np.max(true_accuracies):.4f}]")
print(f"  Reported range from {N_ITERATIONS} iterations: [{np.min(true_accuracies):.4f}, {np.max(true_accuracies):.4f}]")

print(f"\nNULL DISTRIBUTION ({len(null_accuracies_all)} permutation results):")
print(f"  Mean: {null_mean:.4f} ({null_mean*100:.2f}%)")
print(f"  Std:  {null_std:.4f}")

print(f"\nP-VALUES:")
print(f"  Per iteration: {[f'{p:.4f}' for p in p_values]}")
print(f"  Mean: {p_mean:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].plot(range(1, N_ITERATIONS + 1), np.array(true_accuracies) * 100, 'ro-', 
             linewidth=2, markersize=10, label='True Accuracy')
axes[0].axhline(true_mean * 100, color='blue', linestyle='--', linewidth=2, 
               label=f'Mean: {true_mean*100:.2f}%')
axes[0].fill_between(range(1, N_ITERATIONS + 1), 
                     np.min(true_accuracies) * 100, 
                     np.max(true_accuracies) * 100, 
                     alpha=0.2, color='gray', label=f'Actual range: [{np.min(true_accuracies)*100:.2f}%, {np.max(true_accuracies)*100:.2f}%]')
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Accuracy [%]')
axes[0].set_title(f'Observed Accuracy Across {N_ITERATIONS} Iterations')
axes[0].set_xticks(range(1, N_ITERATIONS + 1))
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].hist(np.array(null_accuracies_all) * 100, bins=30, color='purple', alpha=0.7, 
            edgecolor='black', label='Null Distribution')
axes[1].axvline(true_mean * 100, color='red', linewidth=3, 
               label=f'Observed Mean: {true_mean*100:.2f}%')
axes[1].axvline(np.min(true_accuracies) * 100, color='orange', linestyle='--', 
               linewidth=2, label=f'Actual range: [{np.min(true_accuracies)*100:.2f}%, {np.max(true_accuracies)*100:.2f}%]')
axes[1].axvline(np.max(true_accuracies) * 100, color='orange', linestyle='--', linewidth=2)
axes[1].set_xlabel('Accuracy [%]')
axes[1].set_ylabel('Frequency')
axes[1].set_title(f'Null Distribution (n={len(null_accuracies_all)})')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()